# Assignment 3 - Fall Detection using 1D-CNN

The goal of this assignment is to develop a 1D-CNN model in PyTorch to detect falls using the provided [dataset](https://userweb.cs.txstate.edu/~hn12/data/SmartFallDataSet/SmartFall/). The dataset contains 3-axis accelerometer data from a wrist-worn smartwatch. The dataset is split into training and testing sets (files `SmartFall Training.csv` and `SmartFall Testing.csv`; the download cell below saves them as `SmartFall_Training.csv` and `SmartFall_Testing.csv`). Each CSV file contains four columns. The first three columns are the X, Y, and Z accelerometer readings, and the last column is the label (0 for non-fall and 1 for fall). Note that the names of the three accelerometer columns in the header begin with a space (`' ms_accelerometer_x'`, `' ms_accelerometer_y'`, `' ms_accelerometer_z'`), while the label column is named `outcome`. The data is sampled at 31.25 Hz, and each file contains data from multiple users performing various activities and simulated falls. Each row of the CSV file represents a time step and contains 3 accelerometer readings and the corresponding label.
More details about the dataset can be found in this [paper](https://www.mdpi.com/1424-8220/18/10/3363/pdf).

Implement a solution following the steps below.
1. Import the necessary libraries and load the dataset. For convenience, the code to download the dataset is provided below.
2. Use a sliding window approach to segment the time series data.
   - The loaded data should contain a time series of 92,781 time steps in the training set and 91,025 time steps in the test set. To train a deep learning model, we need to segment the time series data into smaller chunks. We will use a sliding window approach to segment the time series data into chunks of 35 time steps each. To better cover the entire time series, we will use a stride of 15 time steps. This means that each 35 time steps window will overlap the previous window by 20 time steps.
   - Before segmenting, pad each time series with 17 rows of zeros (half the window size, `35 // 2`) at the beginning and 17 rows of zeros at the end, and give the padded rows the label 0. The padding lets the first and last time steps of the recording sit at the center of a window instead of only at its edge. With this padding, you should get 6,186 windows from the training set and 6,069 windows from the test set.
   - The sliding window approach will result in array of shape (number_of_windows, 35, 3) for the accelerometer readings and a 1D array of shape (number_of_windows) for the labels. To assign a single label to each window, we will use the most frequent label in the window. If the most frequent label is 1, the label of the window will be 1, otherwise, it will be 0.
3. Convert the data into PyTorch tensors and create a DataLoader. Note that PyTorch 1D-CNN models expect input data in the shape (number_of_windows, channels, sequence_length). In this case, the accelerometer data has 3 channels (X, Y, and Z), and each window has 35 time steps, so the input shape will be (number_of_windows, 3, 35).
   - Hold out a validation set from the training data: train on the first 80% of the training windows and keep the last 20% for validation. Do not split the windows at random. Consecutive windows overlap by 20 time steps, so a random split would put nearly identical windows in both sets and make the validation results look better than they are.
   - Use the validation set to monitor training and to choose hyperparameters (number of epochs, number of filters, dropout rate, etc.). Use the test set only once, for the final evaluation in step 5.
4. Implement a 1D-CNN model in PyTorch that includes at least two 1D convolutional layers with ReLU activations and two linear (fully connected) layers, the last of which is the output layer. You can experiment with other layers, such as batch normalization, max pooling, dropout, etc., to achieve good performance and generalization capability. The model should output the probability of a fall.
5. Classify the test set using a probability threshold of 0.5 and report the accuracy of the model, the confusion matrix, and the full classification report (precision, recall, and F1-score for each class) using the scikit-learn library (`from sklearn.metrics import accuracy_score, classification_report, confusion_matrix`). Your model must achieve an F1-score of at least 0.85 for the fall class (label 1) on the test set. Accuracy alone is not a good measure for this dataset: only about 5.6% of the test windows are falls, so a model that always predicts "no fall" already reaches about 94% accuracy while detecting no falls at all.

In [ ]:
# Download the data
!curl https://userweb.cs.txstate.edu/~hn12/data/SmartFallDataSet/SmartFall/SmartFall%20Training.csv -o SmartFall_Training.csv
!curl https://userweb.cs.txstate.edu/~hn12/data/SmartFallDataSet/SmartFall/SmartFall%20Testing.csv -o SmartFall_Testing.csv